# UK Voting Intention Tracker — Demographic Breakdown Analysis

This notebook loads every poll-tracker workbook in `../data`, combines them into one
tidy dataset, and analyses **how support for each party is moving among different
groups** — age, gender, region, past vote, EU referendum vote, and social grade.

For every party it identifies:
- the group it's currently **strongest** and **weakest** with,
- the group where it's **gaining** ground fastest, and
- the group where it's **losing** ground fastest,

over several trailing windows at once (default: **4, 12 and 52 weeks**), so you can
compare short-term momentum against the medium- and longer-term trend.

## Getting new data

Run the optional cell right below this one to download the latest workbook straight
from YouGov -- no manual downloading or moving files required. It's safe to run every
time: if there's no new poll since your last download it does nothing.

If you'd rather add a file manually (e.g. a workbook from a different source, such as
the pre-2025 archive), just drop it into `../data/` instead and skip that cell -- the
loader picks up every `.xlsx` there automatically. See `data/README.md` for details.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from IPython.display import display, Markdown

from voting_intention import loader, analysis, plotting, downloader

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
%matplotlib inline


**Optional:** fetch the latest tracker workbook from YouGov before analysing. Skip this
cell if you're offline, or if `../data/voting-intention.xlsx` is already current.


In [ ]:
downloader.download_latest(data_dir="../data")


## 1. Load the data

In [ ]:
DATA_DIR = "../data"

# Trailing windows (in weeks) used for "gaining / losing ground" calculations.
# Pass a single number anywhere below (e.g. 12) or a list like this one --
# every analysis/plotting function accepts either.
TRAILING_WEEKS = [4, 12, 52]

df = loader.load_all(DATA_DIR)
print(f"{len(df):,} rows | {df['category'].nunique()} breakdown categories | "
      f"{df['group'].nunique()} groups | {df['party'].nunique()} parties")
df.head()


In [ ]:
# Which file(s) contributed what date range? Also a quick sanity check for gaps
# between files (e.g. the mid-2024 to Jan-2025 hiatus between tracker waves).
coverage = loader.coverage_report(df)
display(coverage)

overall = df[(df["category"] == "Overall")].sort_values("date")
gap_weeks = overall["date"].diff().dt.days / 7
gaps = overall.loc[gap_weeks > 3, ["date"]].assign(gap_weeks=gap_weeks[gap_weeks > 3].round(1))
if not gaps.empty:
    print("Gaps of more than 3 weeks between consecutive polls (won't be bridged by a line in charts):")
    display(gaps)
else:
    print("No gaps of more than 3 weeks detected.")


## 2. National headline trend

In [ ]:
fig = plotting.plot_overall_trend(df, save_path="../outputs/figures/overall_trend.png")


## 3. Build the summary table

One row per (breakdown category, group, party) with the latest support, change since
the data began, a linear trend (pp/month), and -- because `TRAILING_WEEKS` is a list --
a separate `change_4w`, `change_12w` and `change_52w` column all in the same table.


In [ ]:
summary = analysis.build_summary(df, trailing_weeks=TRAILING_WEEKS)
summary.to_csv("../outputs/tables/summary_all_groups.csv", index=False)
print([c for c in summary.columns if c.startswith("change_")])
summary.head()


## 4. Headline: strongest / weakest / gaining / losing, per party

**All breakdowns combined** (including "Past Vote" — so a party's strongest group is
often, unsurprisingly, people who voted for it last time; that's a useful loyalty/
retention signal in its own right). "Strongest"/"weakest" don't depend on the window;
"gaining"/"losing" are shown for each window in `TRAILING_WEEKS` side by side.


In [ ]:
headline_all = analysis.headline_table(summary, trailing_weeks=TRAILING_WEEKS)
headline_all.to_csv("../outputs/tables/headline_all_breakdowns.csv")
display(headline_all)


**Demographics only** (age, gender, region, social grade, EU referendum vote) —
excludes "Past Vote" so you can see genuine demographic patterns rather than simple
vote-retention.


In [ ]:
headline_demographics = analysis.headline_table(
    summary, trailing_weeks=TRAILING_WEEKS, exclude_categories=("Overall", "Past Vote")
)
headline_demographics.to_csv("../outputs/tables/headline_demographics_only.csv")
display(headline_demographics)


## 5. Loyalty / retention: how much of each party's past vote is holding up?

Uses the "Past Vote" breakdown directly: for people who say they voted for party X at
the last election, what share now say they'd vote for X again -- and how has that
retention rate moved over each window?


In [ ]:
PAST_VOTE_LABELS = {"Con": "Conservative", "Lab": "Labour", "Lib Dem": "Liberal Democrat", "Reform UK": "Reform UK"}

past_vote_summary = summary[summary["category"] == "Past Vote"]
retained = past_vote_summary[
    past_vote_summary.apply(lambda r: r["group"] == PAST_VOTE_LABELS.get(r["party"]), axis=1)
].copy()

change_cols = [f"change_{w}w" for w in TRAILING_WEEKS]
retained["retention_%"] = (retained["latest"] * 100).round(1)
for c in change_cols:
    retained[c.replace("change_", "") + "_pp"] = (retained[c] * 100).round(1)

retention = retained.set_index("party")[
    ["latest_date", "retention_%", *[c.replace("change_", "") + "_pp" for c in change_cols], "n_polls"]
]
retention.to_csv("../outputs/tables/vote_retention.csv")
display(retention)


## 6. Breakdown-by-breakdown: trend charts + leaderboards

For each demographic breakdown category: a small-multiples trend chart (one panel per
group, one line per party), followed by a leaderboard table showing each party's
strongest / weakest group within that category, plus its most-gaining/losing group for
every window in `TRAILING_WEEKS`.


In [ ]:
CATEGORIES = ["Age", "Gender", "Region", "Social Grade", "EU Ref Vote", "Past Vote"]

for category in CATEGORIES:
    display(Markdown(f"### {category}"))
    fname = category.lower().replace(' ', '_')
    fig = plotting.plot_category_grid(df, category, save_path=f"../outputs/figures/grid_{fname}.png")
    plotting.plt.show()

    board = analysis.category_leaderboard(summary, category, trailing_weeks=TRAILING_WEEKS)
    board.to_csv(f"../outputs/tables/leaderboard_{fname}.csv")
    display(board)


## 7. At-a-glance heatmaps (all breakdowns together)

**Latest support** — colour shows how strong/weak each group is *relative to that
party's own average* (so parties at very different overall levels are still
comparable); the printed number is always the real percentage.


In [ ]:
fig = plotting.plot_latest_heatmap(summary, save_path="../outputs/figures/latest_heatmap.png")


**Change across all three windows side by side** — percentage points, one panel per
window in `TRAILING_WEEKS`. Reading left to right shows whether a move is brand new
(shows up at 4 weeks but not 52), accelerating (bigger at 4 weeks than the 52-week
average pace would suggest), or fading (large over 52 weeks but flat lately).


In [ ]:
fig = plotting.plot_change_heatmap_grid(
    summary, trailing_weeks=TRAILING_WEEKS, save_path="../outputs/figures/change_heatmap_grid.png"
)


## 8. Momentum for a single party/category

`analysis.momentum_table` is a quick way to inspect one party's movement across every
group in a category, across all windows -- useful once the headline tables above have
pointed you at something interesting. Example: Reform UK by region.


In [ ]:
display(analysis.momentum_table(summary, category="Region", party="Reform UK", trailing_weeks=TRAILING_WEEKS))


## 9. Recap: what's saved to `../outputs/`

- `outputs/tables/summary_all_groups.csv` — the full per-group-per-party summary table
  (one `change_Xw` column per window in `TRAILING_WEEKS`)
- `outputs/tables/headline_all_breakdowns.csv` / `headline_demographics_only.csv` — the
  strongest/weakest/gaining/losing headline tables, gaining/losing broken out per window
- `outputs/tables/vote_retention.csv` — 2024-vote retention by party, per window
- `outputs/tables/leaderboard_<category>.csv` — one leaderboard per breakdown category
- `outputs/figures/*.png` — every chart above, ready to drop into a report or slide deck

Re-running this notebook after adding new poll data, or after changing `TRAILING_WEEKS`,
regenerates everything above with no other changes required.

**Command-line equivalents**, if you'd rather not open Jupyter each time:

```bash
python scripts/update.py            # fetch latest data + regenerate everything
python scripts/fetch_latest_data.py # just fetch, don't regenerate
python scripts/refresh_outputs.py   # just regenerate, don't fetch
```
